In [3]:
class SerpenoidKinematics3D:
    def __init__(self, num_links=16, link_length=0.05105, base_tf=None):
        self.num_links = num_links
        self.link_length = link_length
        self.base_tf = base_tf if base_tf is not None else np.eye(4)
        
    def get_joint_angles(self, params):
        """
        Maps 3D wave parameters to 16 joint angles.
        
        params: [alpha_y, alpha_z, omega, beta, delta_phase, gamma_y, gamma_z]
        
        alpha_y, alpha_z : Amplitudes for Pitch (odd) and Yaw (even)
        omega            : Spatial Frequency (Shared)
        beta             : Base Phase (moves the wave forward)
        delta_phase      : Phase Offset (Crucial for 3D shape/helix)
        gamma_y, gamma_z : Steering Biases
        """
        alpha_y, alpha_z, omega, beta, delta_phase, gamma_y, gamma_z = params
        q = np.zeros(self.num_links)
        
        for i in range(self.num_links):
            # Even joints (0, 2, 4...) = Z-axis / Yaw
            if i % 2 == 0:
                # Uses base phase 'beta'
                angle = alpha_z * np.sin(omega * i + beta) + gamma_z
                q[i] = angle
                
            # Odd joints (1, 3, 5...) = Y-axis / Pitch
            else:
                # Uses offset phase 'beta + delta_phase'
                # This offset allows for Helical/Corkscrew shapes
                angle = alpha_y * np.sin(omega * i + beta + delta_phase) + gamma_y
                q[i] = angle
                
        return q

    def forward_kinematics_ee(self, params):
        """Standard FK to get End Effector position from parameters."""
        q = self.get_joint_angles(params)
        T = self.base_tf.copy()
        
        for i in range(self.num_links):
            theta = q[i]
            c, s = np.cos(theta), np.sin(theta)
            
            # Rotation Logic
            if i % 2 == 0: # Z-axis (Yaw)
                R_mat = np.array([[c, -s, 0, 0], [s, c, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
            else: # Y-axis (Pitch)
                R_mat = np.array([[c, 0, s, 0], [0, 1, 0, 0], [-s, 0, c, 0], [0, 0, 0, 1]])
                
            trans_x = np.array([[1, 0, 0, self.link_length], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
            T = T @ R_mat @ trans_x
            
        return T[0:3, 3]

    def calculate_jacobian(self, params, epsilon=1e-5):
        """
        Computes Jacobian (sensitivity) of End Effector position w.r.t parameters.
        Returns J of shape (3, 7).
        """
        current_ee = self.forward_kinematics_ee(params)
        J = np.zeros((3, len(params)))
        
        for i in range(len(params)):
            p_perturbed = params.copy()
            p_perturbed[i] += epsilon
            ee_perturbed = self.forward_kinematics_ee(p_perturbed)
            J[:, i] = (ee_perturbed - current_ee) / epsilon
            
        return J
    
    def compute_parameter_update(self, current_params, goal_pos, step_size=0.1):
        """
        Returns the optimal parameter change to reach the goal using Damped Least Squares.
        """
        current_ee = self.forward_kinematics_ee(current_params)
        error_vec = goal_pos - current_ee
        
        J = self.calculate_jacobian(current_params)
        
        # Damped Least Squares
        damping = 0.01
        H = J @ J.T + damping * np.eye(3)
        inv_H = np.linalg.inv(H)
        delta_p = J.T @ inv_H @ error_vec
        
        # Clip step size
        norm = np.linalg.norm(delta_p)
        if norm > step_size:
            delta_p = delta_p / norm * step_size
            
        return current_params + delta_p, delta_p, np.linalg.norm(error_vec)




In [1]:
import numpy as np

In [4]:
serp = SerpenoidKinematics3D()

In [ ]:
params = []